# 🤖 Agent Development - Risk Analyst & Compliance Officer

Welcome to Phase 2 & 3 of the project! In this notebook, you'll develop and test both AI agents:

1. **Risk Analyst Agent** (Chain-of-Thought prompting)
2. **Compliance Officer Agent** (ReACT prompting)

## 🎯 Learning Objectives
- Implement Chain-of-Thought prompting for systematic reasoning
- Build ReACT prompting for structured action-taking
- Handle structured JSON outputs from LLMs
- Create robust error handling and validation
- Test agents with real financial data

## 🚀 Setup and Environment

In [1]:
# Import required libraries
import os
import sys
import json
import openai
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime

# Project paths: src/ for the agent modules, the project root for the `src` package helpers
project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
for path in (src_path, project_root):
    if path not in sys.path:
        sys.path.insert(0, path)

# Load environment variables
load_dotenv(os.path.join(project_root, '.env'))
MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

print("📚 Libraries loaded!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path:", src_path)
print(f"🤖 Model: {MODEL}")

📚 Libraries loaded!
🔐 Environment variables loaded
📂 Source directory added to Python path: /Users/maxlucaskienast/Desktop/Education/AI/Agentic_AI/Agentic_AI_Finance_Udacity/1_Prompting/cd14685-fin-serv-agentic-c1-classroom/project/starter/src
🤖 Model: gpt-4o-mini


In [2]:
# OpenAI Setup - Vocareum endpoint for voc- keys (Udacity workspace), standard OpenAI otherwise
from src import create_openai_client

client = create_openai_client()

✅ OpenAI client initialized (OpenAI key)
📍 Base URL: https://api.openai.com/v1 (default)


## 📊 Phase 1 Review: Load Foundation Components

Before building agents, let's ensure your foundation components are working.

In [3]:
# Import the implemented foundation components and the Risk Analyst Agent
from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)
from risk_analyst_agent import RiskAnalystAgent, RISK_ANALYST_SYSTEM_PROMPT
from test_scenarios import RiskAnalystScenarios

print("✅ Foundation components imported")
print("✅ RiskAnalystAgent imported")

✅ Foundation components imported
✅ RiskAnalystAgent imported


In [4]:
# Build real cases for agent testing - one per typology found in Phase 1, plus a benign control
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
audit_log_path = os.path.join(project_root, 'outputs', 'audit_logs', f'phase2_risk_analyst_{run_stamp}.jsonl')
explainability_logger = ExplainabilityLogger(audit_log_path)
loader = DataLoader(explainability_logger)

customers_df, accounts_df, transactions_df = load_csv_data(os.path.join(project_root, 'data'))
account_records = accounts_df.to_dict('records')
transaction_records = transactions_df.to_dict('records')

# Customers identified by the targeted typology rules in 01_data_exploration.ipynb
evaluation_customers = {
    'CUST_0053': 'Structuring (7 cash deposits of $9.2k-$9.9k in 7 days)',
    'CUST_0062': 'Rapid movement (incoming $98k wire fanned out to 4 counterparties)',
    'CUST_0118': 'Cash velocity (cash/ATM activity across 6 branches in days)',
    'CUST_0125': 'Large cash (3 cash transactions of $22k-$45k)',
    'CUST_0002': 'Benign control (no rule triggered, low risk)'
}

real_cases = {}
for customer_id in evaluation_customers:
    customer_record = customers_df[customers_df['customer_id'] == customer_id].iloc[0].to_dict()
    real_cases[customer_id] = loader.create_case_from_data(customer_record, account_records, transaction_records)

print(f"📝 Audit log: {os.path.relpath(audit_log_path, project_root)}")
for customer_id, case in real_cases.items():
    print(f"📁 {customer_id} {case.customer.name:<18} {len(case.accounts)} account(s), "
          f"{len(case.transactions):>3} transactions - {evaluation_customers[customer_id]}")

📝 Audit log: outputs/audit_logs/phase2_risk_analyst_20261011_104240.jsonl
📁 CUST_0053 Tracy Lewis        3 account(s),  94 transactions - Structuring (7 cash deposits of $9.2k-$9.9k in 7 days)
📁 CUST_0062 Michael Stanley    3 account(s),  90 transactions - Rapid movement (incoming $98k wire fanned out to 4 counterparties)
📁 CUST_0118 Patrick Williams   3 account(s),  79 transactions - Cash velocity (cash/ATM activity across 6 branches in days)
📁 CUST_0125 Kayla Jackson      1 account(s),  14 transactions - Large cash (3 cash transactions of $22k-$45k)
📁 CUST_0002 Renee Blair        1 account(s),  11 transactions - Benign control (no rule triggered, low risk)


## 🔍 Phase 2: Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### 📚 Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear phases
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [5]:
# Inspect the Chain-of-Thought system prompt implemented in risk_analyst_agent.py
required_elements = {
    'Senior Financial Crime Risk Analyst persona': 'Senior Financial Crime Risk Analyst',
    'Regulatory context (CTR, structuring, SAR, OFAC)': '31 U.S.C. 5324',
    'Step 1 - Data Review': 'Step 1 - Data Review',
    'Step 2 - Pattern Recognition': 'Step 2 - Pattern Recognition',
    'Step 3 - Regulatory Mapping': 'Step 3 - Regulatory Mapping',
    'Step 4 - Risk Quantification': 'Step 4 - Risk Quantification',
    'Step 5 - Classification Decision': 'Step 5 - Classification Decision',
    'All 5 categories defined': 'Money_Laundering:',
    'Risk level definitions': 'RISK LEVELS',
    'Confidence calibration': 'CONFIDENCE SCORE',
    'JSON output contract': '"key_indicators"',
}

print("🧠 Chain-of-Thought Prompt Structure")
print("=" * 50)
for element, marker in required_elements.items():
    print(f"{'✅' if marker in RISK_ANALYST_SYSTEM_PROMPT else '❌'} {element}")
print(f"\n📏 Prompt length: {len(RISK_ANALYST_SYSTEM_PROMPT):,} characters (~{len(RISK_ANALYST_SYSTEM_PROMPT) // 4:,} tokens)")
print("\n" + "-" * 50)
print(RISK_ANALYST_SYSTEM_PROMPT)

🧠 Chain-of-Thought Prompt Structure
✅ Senior Financial Crime Risk Analyst persona
✅ Regulatory context (CTR, structuring, SAR, OFAC)
✅ Step 1 - Data Review
✅ Step 2 - Pattern Recognition
✅ Step 3 - Regulatory Mapping
✅ Step 4 - Risk Quantification
✅ Step 5 - Classification Decision
✅ All 5 categories defined
✅ Risk level definitions
✅ Confidence calibration
✅ JSON output contract

📏 Prompt length: 5,704 characters (~1,426 tokens)

--------------------------------------------------
You are a Senior Financial Crime Risk Analyst in a U.S. bank's Financial Intelligence Unit, with 15 years of experience in BSA/AML transaction monitoring, alert investigation and SAR decisioning. You analyze cases objectively, cite specific evidence (dates, amounts, transaction types, locations) and never speculate beyond the data provided.

TASK
Classify the suspicious activity in the case into exactly one category, quantify the risk, and explain your decision using the Chain-of-Thought framework below. Your

In [6]:
# Risk Analyst Agent - SIMPLE SMOKE TEST

def simple_risk_analyst_smoke_test():
    """Run one live analysis on a small synthetic structuring case and check the output structure"""
    print("🔍 Risk Analyst Smoke Test")
    try:
        agent = RiskAnalystAgent(client, explainability_logger, model=MODEL)
        test_case = RiskAnalystScenarios().create_case_data('structuring_classic')
        result = agent.analyze_case(test_case)

        assert isinstance(result, RiskAnalystOutput), "Result must be a RiskAnalystOutput"
        assert 0.0 <= result.confidence_score <= 1.0, "Confidence must be within 0-1"
        assert result.reasoning and result.key_indicators, "Reasoning and indicators must be populated"

        print(f"✅ SUCCESS: Got classification '{result.classification}' with confidence {result.confidence_score:.2f} "
              f"(risk level {result.risk_level})")
        print(f"🔑 Indicators: {result.key_indicators}")
        return result
    except Exception as e:
        print(f"❌ FAILED: {e}")
        raise

smoke_result = simple_risk_analyst_smoke_test()

🔍 Risk Analyst Smoke Test
✅ SUCCESS: Got classification 'Structuring' with confidence 0.90 (risk level High)
🔑 Indicators: ['3 cash deposits of $9,800, $9,500, $9,900 over 3 days', 'Total cash deposits of $29,200 just below CTR threshold']


### 🧪 Risk Analyst Testing Framework

In [7]:
# COMPREHENSIVE Risk Analyst Testing - run the pre-built pytest suite from the project root
import inspect
import subprocess

tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

# Preview of the available tests
from test_risk_analyst import TestRiskAnalystAgent
test_methods = [name for name in dir(TestRiskAnalystAgent) if name.startswith('test_')]
print(f"📊 {len(test_methods)} Risk Analyst tests:")
for name in test_methods:
    print(f"   • {getattr(TestRiskAnalystAgent, name).__doc__}")

# Run in a subprocess so pytest's import handling matches the command line exactly
print("\n🚀 Running Risk Analyst test suite...")
pytest_run = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_risk_analyst.py', '-v', '-p', 'no:cacheprovider', '--color=no'],
    cwd=project_root, capture_output=True, text=True
)
print('\n'.join(line for line in pytest_run.stdout.splitlines() if '::' in line or 'passed' in line or 'failed' in line))
print("✅ All Risk Analyst tests passed!" if pytest_run.returncode == 0 else "❌ Some tests failed. Check the output above for details.")

📊 10 Risk Analyst tests:
   • Test RiskAnalystAgent initializes properly
   • Test handling of invalid JSON response
   • Test successful case analysis with valid response
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   • Test JSON extraction from plain text response
   • Test account formatting for prompts
   • Test transaction formatting for prompts
   • Test system prompt contains required elements

🚀 Running Risk Analyst test suite...
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization PASSED [ 10%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success PASSED [ 20%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_json_error PASSED [ 30%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_extract_json_from_code_block PASSED [ 40%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_extract_json_from_plain_text PASSED [ 50%]

### 🔬 Prompt Inspection: What the Agent Sees

`_format_case_for_prompt` turns a `CaseData` object into the user message:
- **Customer profile:** the SSN fragment and address are deliberately left out, because classification doesn't need them.
- **Accounts** and every **transaction**, in chronological order.
- **Pre-computed financial summary:** cash volume, near-threshold deposits, wire flows, activity on closed accounts, and description contradictions.

The summary does the arithmetic up front, so the model spends its reasoning on *interpretation* rather than adding up 94 transactions.

In [8]:
# Show the user prompt built for the structuring case
inspection_agent = RiskAnalystAgent(client, explainability_logger, model=MODEL)
user_prompt = inspection_agent._format_case_for_prompt(real_cases['CUST_0053'])
prompt_lines = user_prompt.splitlines()

print(f"📏 User prompt: {len(prompt_lines)} lines, {len(user_prompt):,} characters (~{len(user_prompt) // 4:,} tokens)")
print("=" * 70)
print('\n'.join(prompt_lines[:30]))
print(f"   ... {len(prompt_lines) - 38} more transactions ...")
print('\n'.join(prompt_lines[-8:]))

📏 User prompt: 121 lines, 10,941 characters (~2,735 tokens)
SUSPICIOUS ACTIVITY CASE 4b096e99-eb94-472c-8a0a-82fba60bc2fe

CUSTOMER PROFILE
- Name: Tracy Lewis (CUST_0053)
- Age: 58 (born 1966-10-26)
- Occupation: Nurse, mental health
- Annual income: $131,925
- Customer since: 2024-05-11 (1 year)
- Internal risk rating: Medium

ACCOUNTS (3)
- CUST_0053_ACC_1 (Money_Market, Active): current balance $17,676.35, average monthly balance $89,530.85, opened 2024-07-23
- CUST_0053_ACC_2 (Savings, Active): current balance $62,101.19, average monthly balance $132,766.49, opened 2024-08-20
- CUST_0053_ACC_3 (Business_Checking, Active): current balance $90,679.35, average monthly balance $193,505.85, opened 2024-11-10

FINANCIAL SUMMARY
- Period: 2025-02-06 to 2025-07-31, 94 transactions, total volume $274,025.18
- Total volume is 2.1x reported annual income
- Volume by type: Cash_Deposit 7 txns $67,569.06; ACH_Credit 11 txns $30,509.50; ATM_Withdrawal 10 txns $29,375.45; ACH_Debit 10 txns $27,9

### 🧠 Chain-of-Thought in Action

The agent asks the model to write out all five steps *before* committing to a decision, and only then emit the JSON. The step-by-step trace is kept (`agent.last_reasoning_trace`) and written to the audit log next to the structured result, so an examiner can follow exactly how each classification was reached.

In [9]:
# Full Chain-of-Thought analysis of the real structuring case
risk_agent = RiskAnalystAgent(client, explainability_logger, model=MODEL)
structuring_result = risk_agent.analyze_case(real_cases['CUST_0053'])

print("🧠 REASONING TRACE")
print("=" * 70)
print(risk_agent.last_reasoning_trace)
print("\n📦 STRUCTURED OUTPUT (validated RiskAnalystOutput)")
print("=" * 70)
print(json.dumps(structuring_result.model_dump(), indent=2))

🧠 REASONING TRACE
CHAIN-OF-THOUGHT ANALYSIS

Step 1 - Data Review: The customer, Tracy Lewis, is a nurse with an annual income of $131,925 and has been a customer for just over a year. She has three active accounts with significant balances, indicating a high volume of transactions totaling $274,025.18 over six months, which is 2.1 times her annual income.

Step 2 - Pattern Recognition: There are 7 cash deposits between $9,000 and $10,000 totaling $67,569.06 within a week (2025-07-25 to 2025-07-31), which raises red flags for potential structuring to avoid CTR reporting. Additionally, there are multiple cash deposits made at different branches.

Step 3 - Regulatory Mapping: The repeated cash deposits just under the $10,000 threshold indicate structuring, as they appear designed to evade CTR reporting. Other categories like sanctions, fraud, and money laundering do not apply based on the provided data.

Step 4 - Risk Quantification: The total cash deposits are significant relative to th

### 📊 Real-Data Evaluation

The same agent run on the other four real cases: one per remaining typology from Phase 1, plus a benign control. The Phase 1 rules give a behavioural *expectation*. The agent has to reach its own conclusion from the raw case data.

In [10]:
# Analyze the remaining real cases and compare with the Phase 1 rule findings
real_results = {'CUST_0053': structuring_result}
for customer_id, case in real_cases.items():
    if customer_id not in real_results:
        real_results[customer_id] = risk_agent.analyze_case(case)

evaluation_table = pd.DataFrame([
    {
        'customer': f"{customer_id} {real_cases[customer_id].customer.name}",
        'phase_1_finding': evaluation_customers[customer_id].split(' (')[0],
        'classification': result.classification,
        'confidence': result.confidence_score,
        'risk_level': result.risk_level,
        'key_indicators': '; '.join(result.key_indicators[:3])
    }
    for customer_id, result in real_results.items()
])
print("📊 Real-Data Risk Analysis")
print("=" * 70)
print(evaluation_table.to_string(index=False))

print("\n💬 Condensed reasoning per case:")
for customer_id, result in real_results.items():
    print(f"\n🔸 {customer_id} ({result.classification}, {result.risk_level}):\n   {result.reasoning}")

📊 Real-Data Risk Analysis
                  customer phase_1_finding   classification  confidence risk_level                                                                                                                                                                                  key_indicators
     CUST_0053 Tracy Lewis     Structuring      Structuring        0.90       High                                                                                7 cash deposits of $9,000-$9,915.61 across 5 branches in 7 days; Total cash deposits of $67,569.06 within a week
 CUST_0062 Michael Stanley  Rapid movement Money_Laundering        0.90       High Incoming wire of $98,090.16 followed by rapid outflows; Outflows totaling $24,082.60, $17,747.14, $10,516.73, $18,835.61 within 2 days; Contradictory description for incoming wire transaction
CUST_0118 Patrick Williams   Cash velocity      Structuring        0.85       High                                                    Multiple cash d

### 🧪 Synthetic Scenario Suite: All 5 Classification Types

The CSV extract has no sanctions exposure (Phase 1 finding). The pre-built scenarios in `src/test_scenarios.py` cover all five categories, including Sanctions, elder-fraud and a benign control, each with an expected classification, risk level and minimum confidence.

*`test_scenarios.py` was updated to match the implemented schemas. It previously used `date` instead of `transaction_date`, omitted the required `method` field, and built cases without `case_created_at`/`data_sources`.*

In [11]:
# Run all pre-built Risk Analyst scenarios
scenario_suite = RiskAnalystScenarios()
scenario_summary = scenario_suite.run_all_scenarios(risk_agent)

scenario_table = pd.DataFrame([
    {
        'scenario': name,
        'expected': result.get('expected_classification'),
        'actual': result.get('actual_classification'),
        'confidence': result.get('confidence_score'),
        'min_confidence': scenario_suite.scenarios[name]['min_confidence'],
        'expected_risk': result.get('expected_risk_level'),
        'actual_risk': result.get('actual_risk_level'),
        'pass': result.get('overall_pass')
    }
    for name, result in scenario_summary['detailed_results'].items()
])
print("\n" + scenario_table.to_string(index=False))

classification_accuracy = (scenario_table['expected'] == scenario_table['actual']).mean()
risk_level_accuracy = (scenario_table['expected_risk'] == scenario_table['actual_risk']).mean()
print(f"\n🎯 Classification accuracy: {classification_accuracy:.0%} | Risk level agreement: {risk_level_accuracy:.0%}")

🧪 Running Risk Analyst Scenario Tests...

🔍 Testing: Classic Structuring Pattern
   ✅ PASS - Classification: Structuring (confidence: 0.90)

🔍 Testing: Complex Money Laundering Pattern
   ✅ PASS - Classification: Money_Laundering (confidence: 0.90)

🔍 Testing: Identity Fraud Pattern
   ✅ PASS - Classification: Fraud (confidence: 0.85)

🔍 Testing: Sanctions Violation
   ✅ PASS - Classification: Sanctions (confidence: 0.95)

🔍 Testing: Benign Business Activity
   ✅ PASS - Classification: Other (confidence: 0.90)

📊 SUMMARY: 5/5 scenarios passed (100.0%)

                scenario         expected           actual  confidence  min_confidence expected_risk actual_risk  pass
     structuring_classic      Structuring      Structuring        0.90            0.80          High        High  True
money_laundering_complex Money_Laundering Money_Laundering        0.90            0.85      Critical        High  True
          fraud_identity            Fraud            Fraud        0.85            0.

### 🛡️ Error Handling & Recovery

LLM output is untrusted input. The agent defends against each failure mode below. They are demonstrated with mocked API responses (no API cost) and logged to a separate demo audit file, so the real audit trail only holds real analyses:

| Failure mode | Handling |
|---|---|
| JSON wrapped in prose or code fences | Extractor searches fenced blocks, then any decodable object |
| Formatting slips (`"money laundering"`, `"85%"`, `"high"`, overlong reasoning, trailing commas) | Normalised or repaired, and every adjustment recorded in the audit entry |
| Sanctions classified below Critical | Policy floor raises the risk level to Critical (sanctions exposure is prohibited, not merely suspicious), and records the adjustment |
| No JSON / invalid schema on the first try | One **corrective retry** that feeds the exact error back to the model |
| Still invalid after retries | `ValueError`, with a failure audit entry |
| Truncated output (`finish_reason == "length"`) | Reported explicitly in the error |
| API failure (network, rate limit) | `RuntimeError`, with a failure audit entry |

In [12]:
# Demonstrate error handling with mocked LLM responses
import tempfile
from unittest.mock import Mock

def mock_response(content, finish_reason='stop'):
    response = Mock()
    response.choices = [Mock()]
    response.choices[0].message.content = content
    response.choices[0].finish_reason = finish_reason
    response.usage = None
    return response

valid_json = json.dumps({"classification": "Structuring", "confidence_score": 0.88,
                         "reasoning": "Profile: ... Pattern: ... Decision: Structuring",
                         "key_indicators": ["7 cash deposits $9.2k-$9.9k", "5 branches"], "risk_level": "High"})
sloppy_json = json.dumps({"classification": "money laundering", "confidence_score": "85%",
                          "reasoning": "Step-by-step analysis " * 40, "key_indicators": "pass-through wires",
                          "risk_level": "high"})
demo_scenarios = {
    'JSON in prose + code fence': [mock_response(f"Step 1 - Data Review: ...\n```json\n{valid_json}\n```\nDone.")],
    'Formatting slips normalised': [mock_response(sloppy_json)],
    'Trailing commas repaired': [mock_response(valid_json.replace('"High"}', '"High",}'))],
    'Corrective retry recovers': [mock_response("The activity looks like structuring."), mock_response(valid_json)],
    'Invalid category twice': [mock_response(valid_json.replace("Structuring", "Terrorism"))] * 2,
    'Truncated output twice': [mock_response('```json\n{"classification": "Struc', 'length')] * 2,
    'API connection failure': openai.APIConnectionError(request=Mock()),
}

demo_logger = ExplainabilityLogger(os.path.join(tempfile.mkdtemp(), 'error_handling_demo.jsonl'))
demo_case = real_cases['CUST_0053']
demo_rows = []
for label, responses in demo_scenarios.items():
    mock_client = Mock()
    mock_client.chat.completions.create.side_effect = responses
    demo_agent = RiskAnalystAgent(mock_client, demo_logger, model=MODEL)
    try:
        result = demo_agent.analyze_case(demo_case)
        outcome = f"✅ {result.classification} / {result.confidence_score} / {result.risk_level}"
    except (ValueError, RuntimeError) as e:
        outcome = f"🛑 {type(e).__name__}: {str(e).splitlines()[0][:70]}"
    entry = demo_logger.entries[-1]
    demo_rows.append({'scenario': label, 'api_calls': mock_client.chat.completions.create.call_count,
                      'outcome': outcome, 'audit_entry': f"success={entry['success']} | {entry['reasoning'][:45]}"})

print(pd.DataFrame(demo_rows).to_string(index=False))

# The adjustments the agent recorded in the audit entry for the sloppy response
_, adjustments = RiskAnalystAgent(Mock(), demo_logger)._normalize_output(json.loads(sloppy_json))
print("\n📝 Normalizations recorded for 'Formatting slips normalised':")
for adjustment in adjustments:
    print(f"   • {adjustment}")
assert len(demo_logger.entries) == len(demo_scenarios), "Exactly one audit entry per analysis"
print(f"\n✅ {len(demo_logger.entries)} analyses -> {len(demo_logger.entries)} audit entries (one per analysis, success or failure)")

                   scenario  api_calls                                                                              outcome                                                   audit_entry
 JSON in prose + code fence          1                                                          ✅ Structuring / 0.88 / High  success=True | Profile: ... Pattern: ... Decision: Structuri
Formatting slips normalised          1                                                     ✅ Money_Laundering / 0.85 / High  success=True | Step-by-step analysis Step-by-step analysis S
   Trailing commas repaired          1                                                          ✅ Structuring / 0.88 / High  success=True | Profile: ... Pattern: ... Decision: Structuri
  Corrective retry recovers          2                                                          ✅ Structuring / 0.88 / High  success=True | Profile: ... Pattern: ... Decision: Structuri
     Invalid category twice          2 🛑 ValueError: Risk Analyst outp

### 📝 Audit Trail & Cost Review

Every live analysis above was written to `outputs/audit_logs/`, together with the case-creation entries from the `DataLoader`. Each Risk Analyst entry includes the validated decision, the full Chain-of-Thought trace, any normalisations, the attempt count and token usage. That is the basis for the Phase 4 cost metrics.

In [13]:
# Review the Phase 2 audit trail and token usage
with open(audit_log_path) as f:
    audit_entries = [json.loads(line) for line in f]

analyst_entries = [e for e in audit_entries if e['agent_type'] == 'RiskAnalyst']
print("📝 Audit Trail Summary")
print("=" * 50)
print(f"📄 File: {os.path.relpath(audit_log_path, project_root)}")
print(f"🧾 Entries: {len(audit_entries)} total - "
      f"{sum(e['agent_type'] == 'DataLoader' for e in audit_entries)} DataLoader, {len(analyst_entries)} RiskAnalyst")
print(f"✅ Successful analyses: {sum(e['success'] for e in analyst_entries)} / {len(analyst_entries)}")
print(f"⏱️  Avg analysis latency: {sum(e['execution_time_ms'] for e in analyst_entries) / len(analyst_entries) / 1000:.1f} s")

# Token usage straight from the audit trail - each RiskAnalyst entry records its own usage and attempts
import ast
usage = {'api_calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0}
for entry in analyst_entries:
    details = ast.literal_eval(entry['output_summary'])
    usage['api_calls'] += details['attempts']
    usage['prompt_tokens'] += details['token_usage']['prompt_tokens']
    usage['completion_tokens'] += details['token_usage']['completion_tokens']

# gpt-4o-mini list prices in USD per 1M tokens - update if the model or pricing changes
PRICE_PER_1M = {'prompt_tokens': 0.15, 'completion_tokens': 0.60}
estimated_cost = sum(usage[key] / 1_000_000 * price for key, price in PRICE_PER_1M.items())
print(f"\n💰 Token usage ({usage['api_calls']} API calls for this run): "
      f"{usage['prompt_tokens']:,} prompt + {usage['completion_tokens']:,} completion tokens")
print(f"💵 Estimated cost: ${estimated_cost:.4f} (~${estimated_cost / max(usage['api_calls'], 1):.5f} per analysis)")

print("\n🔍 Sample RiskAnalyst audit entry:")
sample_entry = dict(analyst_entries[0])
sample_entry['output_summary'] = sample_entry['output_summary'][:600] + '...'
print(json.dumps(sample_entry, indent=2))

📝 Audit Trail Summary
📄 File: outputs/audit_logs/phase2_risk_analyst_20261011_104240.jsonl
🧾 Entries: 16 total - 5 DataLoader, 11 RiskAnalyst
✅ Successful analyses: 11 / 11
⏱️  Avg analysis latency: 3.6 s

💰 Token usage (11 API calls for this run): 29,359 prompt + 4,759 completion tokens
💵 Estimated cost: $0.0073 (~$0.00066 per analysis)

🔍 Sample RiskAnalyst audit entry:
{
  "timestamp": "2026-10-11T09:42:44.681040+00:00",
  "case_id": "CASE_STRUCTURING_CLASSIC",
  "agent_type": "RiskAnalyst",
  "action": "analyze_case",
  "input_summary": "{'customer_id': 'CUST_STRUCT_001', 'num_accounts': 1, 'num_transactions': 3, 'model': 'gpt-4o-mini'}",
  "output_summary": "{'classification': 'Structuring', 'confidence_score': 0.9, 'reasoning': 'Profile: Customer has no reported income or occupation. Pattern: 3 cash deposits of $9,800, $9,500, and $9,900 over 3 days. Mapping: This behavior indicates Structuring to evade CTR reporting. Risk: High due to repeated threshold avoidance. Decision: Clas

## ✅ Phase 3: Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

### 📚 Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [14]:
# TODO: Test ReACT prompt design
# This cell helps you design and test your ReACT prompt structure

def design_react_prompt():
    """Design and test ReACT prompt for compliance narratives"""
    
    system_prompt = """
    TODO: Design your ReACT system prompt here
    
    Key elements to include:
    1. Senior Compliance Officer persona
    2. ReACT framework:
       REASONING Phase:
       - Review risk analyst findings
       - Assess regulatory requirements
       - Identify compliance elements
       - Plan narrative structure
       
       ACTION Phase:
       - Draft concise narrative (≤120 words)
       - Include specific details
       - Reference activity patterns
       - Use regulatory language
    3. JSON output format specification
    """
    
    return system_prompt

# Test your prompt design
react_prompt = design_react_prompt()
print("⚡ ReACT Prompt Design:")
print(react_prompt[:200] + "...")
print("\n📋 TODO: Complete the prompt in compliance_officer_agent.py")

⚡ ReACT Prompt Design:

    TODO: Design your ReACT system prompt here

    Key elements to include:
    1. Senior Compliance Officer persona
    2. ReACT framework:
       REASONING Phase:
       - Review risk analyst find...

📋 TODO: Complete the prompt in compliance_officer_agent.py


In [15]:
# TODO: Implement and test Compliance Officer Agent - SIMPLE SMOKE TEST
# Students: Write a basic smoke test to verify your agent works

# from compliance_officer_agent import ComplianceOfficerAgent

def simple_compliance_officer_smoke_test():
    """
    STUDENT TODO: Write a simple smoke test for your Compliance Officer Agent
    
    This should be a basic test that:
    1. Creates a ComplianceOfficerAgent instance
    2. Creates simple test case and risk analysis data
    3. Calls generate_compliance_narrative() method  
    4. Verifies the result has a narrative with reasonable length
    5. Prints success/failure
    
    Keep this simple - just verify your agent doesn't crash and generates text!
    """
    print("✅ Compliance Officer Smoke Test")
    print("📋 TODO: Import your ComplianceOfficerAgent")
    print("📋 TODO: Create agent instance: agent = ComplianceOfficerAgent(client, logger)")
    print("📋 TODO: Create simple test case and sample risk analysis")
    print("📋 TODO: Call: result = agent.generate_compliance_narrative(case, risk_analysis)")
    print("📋 TODO: Verify: result has narrative, word count ≤ 120")
    print("📋 TODO: Print: 'SUCCESS' or 'FAILED' with details")
    
    # Example structure (uncomment and modify when ready):
    # try:
    #     agent = ComplianceOfficerAgent(client, explainability_logger)
    #     # Create test case and risk analysis...
    #     result = agent.generate_compliance_narrative(test_case, test_risk_analysis)
    #     word_count = len(result.narrative.split())
    #     print(f"✅ SUCCESS: Generated {word_count} word narrative")
    #     print(f"Preview: {result.narrative[:100]}...")
    # except Exception as e:
    #     print(f"❌ FAILED: {e}")

simple_compliance_officer_smoke_test()

✅ Compliance Officer Smoke Test
📋 TODO: Import your ComplianceOfficerAgent
📋 TODO: Create agent instance: agent = ComplianceOfficerAgent(client, logger)
📋 TODO: Create simple test case and sample risk analysis
📋 TODO: Call: result = agent.generate_compliance_narrative(case, risk_analysis)
📋 TODO: Verify: result has narrative, word count ≤ 120
📋 TODO: Print: 'SUCCESS' or 'FAILED' with details


### 🧪 Compliance Officer Testing Framework

In [16]:
# COMPREHENSIVE Compliance Officer Testing - Import Pre-Built Test Suite
# Students: Use our comprehensive test suite instead of writing your own

import sys
import os

# Add tests directory to Python path for importing test modules (if not already added)
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Tests directory in Python path: {tests_path}")

def run_comprehensive_compliance_officer_tests():
    """
    Use pre-built comprehensive test suite to validate your Compliance Officer Agent
    
    These tests validate:
    - Agent initialization and configuration
    - Narrative generation with valid inputs
    - Word count limits (≤120 words)
    - Regulatory citations inclusion
    - JSON parsing and error handling
    - ReACT prompt structure validation
    """
    print("🧪 Comprehensive Compliance Officer Testing")
    print("📋 TODO: Uncomment and run after implementing your Compliance Officer Agent")
    
    # Uncomment when your agent is ready:
    # try:
    #     from test_compliance_officer import TestComplianceOfficerAgent
    #     import pytest
    #     
    #     print("📝 Loading comprehensive test suite...")
    #     
    #     # Run the test suite
    #     print("🚀 Running Compliance Officer test suite...")
    #     result = pytest.main([
    #         f"{tests_path}/test_compliance_officer.py", 
    #         "-v", 
    #         "--tb=short"
    #     ])
    #     
    #     if result == 0:
    #         print("✅ All Compliance Officer tests passed!")
    #     else:
    #         print("❌ Some tests failed. Check the output above for details.")
    #         
    # except ImportError as e:
    #     print(f"❌ Import Error: {e}")
    #     print("💡 Make sure you've implemented ComplianceOfficerAgent in src/compliance_officer_agent.py")

# Quick preview of available tests
try:
    from test_compliance_officer import TestComplianceOfficerAgent
    import inspect
    
    # Get all test methods
    test_methods = [method for method in dir(TestComplianceOfficerAgent) 
                   if method.startswith('test_')]
    
    print("📝 Preview of Comprehensive Compliance Officer Tests:")
    for method_name in test_methods[:5]:  # Show first 5
        method = getattr(TestComplianceOfficerAgent, method_name)
        doc = method.__doc__ or method_name.replace('_', ' ').title()
        print(f"   • {doc}")
    if len(test_methods) > 5:
        print(f"   ... and {len(test_methods) - 5} more tests")
    print("\n💡 These tests validate regulatory compliance requirements!")
    print("💡 Includes word limits, citations, and required elements!")
except Exception as e:
    print(f"ℹ️ Test suite will be available after implementing foundation_sar.py: {e}")

# Call the function
run_comprehensive_compliance_officer_tests()

📁 Tests directory in Python path: /Users/maxlucaskienast/Desktop/Education/AI/Agentic_AI/Agentic_AI_Finance_Udacity/1_Prompting/cd14685-fin-serv-agentic-c1-classroom/project/starter/tests
📝 Preview of Comprehensive Compliance Officer Tests:
   • Test ComplianceOfficerAgent initializes properly
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   • Test JSON extraction from plain text response
   ... and 5 more tests

💡 These tests validate regulatory compliance requirements!
💡 Includes word limits, citations, and required elements!
🧪 Comprehensive Compliance Officer Testing
📋 TODO: Uncomment and run after implementing your Compliance Officer Agent


In [17]:
# COMPLETE AGENT TESTING - Two-Tier Approach
# Students: Use this to test both agents together

def complete_agent_testing_workflow():
    """
    Complete testing workflow using two-tier approach:
    
    TIER 1: Simple Smoke Tests (You write these)
    - Basic functionality verification
    - Quick sanity checks
    - Development debugging
    
    TIER 2: Comprehensive Test Suites (Pre-built for you)
    - Complex edge cases
    - Regulatory compliance validation
    - Professional-grade testing
    """
    print("🔬 Complete Agent Testing Workflow")
    print("=" * 50)
    
    print("\n📋 TIER 1: Simple Smoke Tests (DO FIRST)")
    print("   1. Write simple_risk_analyst_smoke_test() - verify basic functionality")
    print("   2. Write simple_compliance_officer_smoke_test() - verify basic functionality")
    print("   3. Fix any basic issues before moving to Tier 2")
    
    print("\n🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)")
    print("   1. Run comprehensive risk analyst test suite (10 comprehensive tests)")
    print("   2. Run comprehensive compliance officer test suite (10 comprehensive tests)")
    print("   3. Get detailed pass/fail results with specific feedback")
    
    print("\n💡 WHY THIS APPROACH?")
    print("   ✅ Tier 1: Quick feedback while developing")
    print("   ✅ Tier 2: Professional validation without writing complex tests")
    print("   ✅ Saves time: You focus on implementation, not test creation")
    print("   ✅ Better coverage: Our test suites test edge cases you might miss")

# Quick test runner when both agents are ready
def run_both_agents_quick_test():
    """Quick test of both agents using pre-built test suites"""
    print("🚀 Quick Test of Both Agents")
    print("📋 TODO: Uncomment when both agents are implemented")
    
    # Uncomment when ready:
    # try:
    #     import pytest
    #     
    #     print("🔍 Running quick tests for both agents...")
    #     
    #     # Run a subset of tests for quick validation
    #     risk_result = pytest.main([
    #         f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization",
    #         f"{tests_path}/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success",
    #         "-v"
    #     ])
    #     
    #     compliance_result = pytest.main([
    #         f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization", 
    #         f"{tests_path}/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success",
    #         "-v"
    #     ])
    #     
    #     if risk_result == 0 and compliance_result == 0:
    #         print("🎉 Both agents working! Ready for full test suite testing!")
    #     else:
    #         print("⚠️ Fix issues before running comprehensive tests")
    #         if risk_result != 0:
    #             print("   🔍 Risk Analyst needs fixes")
    #         if compliance_result != 0:
    #             print("   📝 Compliance Officer needs fixes")
    #             
    # except ImportError as e:
    #     print(f"❌ Import Error: {e}")
    #     print("💡 Make sure both agents are implemented")

complete_agent_testing_workflow()
run_both_agents_quick_test()

🔬 Complete Agent Testing Workflow

📋 TIER 1: Simple Smoke Tests (DO FIRST)
   1. Write simple_risk_analyst_smoke_test() - verify basic functionality
   2. Write simple_compliance_officer_smoke_test() - verify basic functionality
   3. Fix any basic issues before moving to Tier 2

🧪 TIER 2: Comprehensive Test Suites (DO AFTER TIER 1 PASSES)
   1. Run comprehensive risk analyst test suite (10 comprehensive tests)
   2. Run comprehensive compliance officer test suite (10 comprehensive tests)
   3. Get detailed pass/fail results with specific feedback

💡 WHY THIS APPROACH?
   ✅ Tier 1: Quick feedback while developing
   ✅ Tier 2: Professional validation without writing complex tests
   ✅ Saves time: You focus on implementation, not test creation
   ✅ Better coverage: Our test suites test edge cases you might miss
🚀 Quick Test of Both Agents
📋 TODO: Uncomment when both agents are implemented


## 🔗 Phase 4 Preview: Agent Integration

Once both agents are working, you'll integrate them into a complete workflow.

In [18]:
# TODO: Preview of integrated workflow
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1. 📊 Load and validate case data",
        "2. 🔍 Risk Analyst performs Chain-of-Thought analysis",
        "3. 👤 Human review and approval gate",
        "4. ✅ Compliance Officer generates ReACT narrative (if approved)",
        "5. 📄 Generate complete SAR document",
        "6. 📊 Log audit trail and efficiency metrics"
    ]
    
    print("🔗 Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n💡 Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

🔗 Integrated SAR Processing Workflow:
1. 📊 Load and validate case data
2. 🔍 Risk Analyst performs Chain-of-Thought analysis
3. 👤 Human review and approval gate
4. ✅ Compliance Officer generates ReACT narrative (if approved)
5. 📄 Generate complete SAR document
6. 📊 Log audit trail and efficiency metrics

💡 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## 📝 Development Checklist - Two-Tier Testing Approach

### ✅ Risk Analyst Agent (Phase 2)
- [x] Implement Chain-of-Thought system prompt
- [x] Create `analyze_case` method with error handling
- [x] Add JSON parsing and validation
- [x] **TIER 1**: Write simple smoke test (verify basic functionality)
- [x] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [x] Fix any issues identified by test suite

### ✅ Compliance Officer Agent (Phase 3)  
- [ ] Implement ReACT system prompt
- [ ] Create `generate_compliance_narrative` method
- [ ] Add narrative validation (word count, terminology)
- [ ] **TIER 1**: Write simple smoke test (verify basic functionality)
- [ ] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [ ] Fix any issues identified by test suite

### ✅ Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### 💡 **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify your agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## 🚀 Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## 📊 Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing! 🕵️‍♀️**